# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Approach:** score every page with the Week-5 Random Forest (`decline_risk`), then group visible pages into 4 archetypes with K-Means so the queue reads as "kinds of pages", not just a bare number. Archetypes + model score together decide the action — a page's *type* can override a raw score (see the Early-Stage rule below), which is the human-judgment layer this assignment asks for.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans

RANDOM_SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["word_count"] = df.groupby("content_type")["word_count"].transform(lambda s: s.fillna(s.median()))
df["main_intent"] = df["main_intent"].fillna("unknown")

numeric_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                     "engagement_rate", "ai_traffic_pct", "content_age_days",
                     "days_since_last_update", "word_count"]
categorical_features = ["content_type", "main_intent"]
X = df[numeric_features + categorical_features]
y = df["is_declining_label"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, df["client_id"]))

rf_prep = ColumnTransformer([("num", "passthrough", numeric_features), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
rf = Pipeline([("prep", rf_prep), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1))])
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
df["decline_risk"] = rf.predict_proba(X)[:, 1]  # score EVERY row now -- this is deployment, not evaluation

# --- Archetypes: K-Means (k=4) on visible pages only -- same idea as the FlyRank paper's
# archetype appendix, but named from OUR OWN cluster centers, not borrowed labels ---
df["archetype"] = "Low-Visibility / Not Clustered"
visible_mask = df["impressions_90d"] >= 100
cluster_features = ["impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update", "word_count"]
Xc = df.loc[visible_mask, cluster_features].copy()
Xc["avg_position"] = Xc["avg_position"].replace(0, Xc["avg_position"].median())  # 0 means "no data"
Xc_scaled = StandardScaler().fit_transform(Xc)
km = KMeans(n_clusters=4, random_state=RANDOM_SEED, n_init=10)
cluster_ids = km.fit_predict(Xc_scaled)

centers = df.loc[visible_mask].assign(cluster=cluster_ids).groupby("cluster")[cluster_features + ["decline_risk"]].mean().round(1)
print("Cluster centers (used to NAME each archetype below):")
print(centers)

name_map = {0: "Early-Stage / Still Settling", 1: "Aging & Under-Refreshed",
            2: "Refreshed Veterans", 3: "High-Traffic Long-Form, Overdue"}
df.loc[visible_mask, "archetype"] = pd.Series(cluster_ids, index=df.loc[visible_mask].index).map(name_map)
print("\nArchetype sizes:")
print(df["archetype"].value_counts())

Cluster centers (used to NAME each archetype below):
         impressions_90d  avg_position  ctr  content_age_days  \
cluster                                                         
0                 5958.5          14.4  0.3             137.7   
1                 6032.7          15.9  0.2             269.8   
2                 5574.8          20.8  0.2             442.3   
3                15606.5          21.4  0.3             232.9   

         days_since_last_update  word_count  decline_risk  
cluster                                                    
0                          19.0      2982.8           0.7  
1                         104.5      2580.0           0.6  
2                          20.3      2804.0           0.5  
3                         102.3      5962.2           0.6  

Archetype sizes:
archetype
Early-Stage / Still Settling       8130
Low-Visibility / Not Clustered     7994
Refreshed Veterans                 5708
Aging & Under-Refreshed            5423
High-Tra

**Naming the archetypes from their actual centers** (not borrowed from FlyRank's paper — these are this dataset's own clusters):

- **Early-Stage / Still Settling** (n=8,130): youngest (age 138d), freshest (updated 19d ago), *best* average position (14.4) of any cluster — yet the *highest* decline risk (0.7). Reading this as "low quality" would be wrong: it's more likely early-lifecycle volatility (FlyRank's own paper, Finding #2, shows the growth phase runs through ~90 days before peak). **Business rule: don't refresh these — they're still stabilizing.**
- **Aging & Under-Refreshed** (n=5,423): moderate traffic, older (270d), stale (105d since update). The textbook refresh candidate.
- **Refreshed Veterans** (n=5,708): the *oldest* pages (442d) but *most recently updated* (20d) — and the *lowest* decline risk (0.5) of any visible cluster. This directly matches FlyRank's own Finding #8 ("old content that gets refreshed performs nearly as well as new content") — a nice independent confirmation from a completely different sample.
- **High-Traffic Long-Form, Overdue** (n=2,745): the smallest group but by far the highest traffic (15,607 impressions) and longest content (5,962 words), sitting at a weak position (21.4) and stale (102d). Small in count, large in reach — refreshing these affects the most readers per page touched.

**Reason codes + final action** (model score is the base signal; archetype membership can override it — this is exactly the human-judgment layer a raw model score can't provide on its own):

In [2]:
def reason_code(row):
    if row["impressions_90d"] < 100: return "low_visibility"
    if row["ctr"] == 0 and row["days_since_last_update"] >= 90: return "stale_and_zero_ctr"
    if row["days_since_last_update"] >= 90: return "stale_visible"
    if row["ctr"] < 0.15: return "ctr_underperforming"
    return "no_flag"
df["reason_code"] = df.apply(reason_code, axis=1)

def final_action(row):
    if row["archetype"] == "Low-Visibility / Not Clustered": return "insufficient_data"
    if row["archetype"] == "Early-Stage / Still Settling": return "monitor_dont_touch"  # archetype OVERRIDES score
    if row["decline_risk"] >= 0.70: return "refresh_priority"
    if row["decline_risk"] >= 0.55: return "refresh_soon"
    if row["decline_risk"] >= 0.40: return "monitor"
    return "no_action"
df["action"] = df.apply(final_action, axis=1)

print(df["action"].value_counts())
print()
print(pd.crosstab(df["archetype"], df["action"]))

ranked = df.sort_values("decline_risk", ascending=False)
print("\nTop 5 of the ranked queue:")
print(ranked[["content_id","archetype","decline_risk","reason_code","action"]].head(5).to_string(index=False))

action
monitor_dont_touch    8130
insufficient_data     7994
monitor               5108
refresh_soon          3754
refresh_priority      3042
no_action             1972
Name: count, dtype: int64

action                           insufficient_data  monitor  \
archetype                                                     
Aging & Under-Refreshed                          0     1195   
Early-Stage / Still Settling                     0        0   
High-Traffic Long-Form, Overdue                  0      554   
Low-Visibility / Not Clustered                7994        0   
Refreshed Veterans                               0     3359   

action                           monitor_dont_touch  no_action  \
archetype                                                        
Aging & Under-Refreshed                           0        316   
Early-Stage / Still Settling                   8130          0   
High-Traffic Long-Form, Overdue                   0         34   
Low-Visibility / Not Clustered  

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a weekly-review shortlist for a content/SEO editor deciding which pages to look at first — not an auto-publish or auto-refresh system. `refresh_priority` and `refresh_soon` mean "put this near the top of someone's reading list," nothing more.

**Who should use it:** an editor or content strategist with authority over the pages in question, working through the queue a page at a time.

**Where it stops being valid:**
- **Sample scope:** trained on 30,000 rows across 32 clients — a teaching slice of FlyRank's much larger warehouse (79M+ rows across 519K content items). Don't assume these exact archetype boundaries hold at that scale.
- **Time scope:** a single 90-day snapshot. Re-check before using it on a portfolio months later — see monitoring triggers below.
- **Content type scope:** the archetypes were shaped mostly by `keyword article` behavior (the dominant content_type in this file); other content types may not fit these 4 buckets well.
- **Client scope:** validated on 8 held-out clients (Week 5/6). A client with a very different publishing pattern (e.g., mostly navigational content, or a brand-new site with no history) may not be well represented.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a human must check before acting on any `refresh_priority` / `refresh_soon` row:**
- Is this page still strategically relevant to the client (not a deprecated product/service)?
- Is the low CTR/position actually a content problem, or a technical one (noindex, wrong canonical, broken redirect)? The Week-5 error analysis found a real case of this — refreshing content doesn't fix an indexing bug.
- Is this a genuinely low-intent query where even great content sits near-zero CTR (Week-5 also found this among the false negatives)?
- Does this page share a topic with another page in the queue (cannibalization)? FlyRank's own paper flags 703K cannibalizing queries portfolio-wide — merging may beat refreshing.

**What should NEVER be automated from this notebook alone:**
- Auto-publishing AI-rewritten content without a human edit pass.
- Auto-deleting, de-indexing, or redirecting a `no_action` page — absence of a flag is not evidence a page is safe to remove.
- Any action on `Early-Stage / Still Settling` pages — the model has no reliable signal here yet, by design (see the archetype override above).
- Bulk actions across an entire archetype without spot-checking a sample first — archetypes are population-level descriptions, not guarantees about any individual page.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring:** recompute the model's precision@50 monthly against that month's actual `trend_direction` outcomes; compare to this run's 0.660 (Week 5, grouped split) and to the base rate. Also track the archetype size distribution — if `Early-Stage / Still Settling` starts shrinking or growing sharply as a share of the portfolio, the cluster boundaries may no longer describe the current content mix.

**Retrain triggers:**
- Precision@50 drops more than ~10 points below this run's 0.660, sustained over 2+ monthly checks.
- A new client is onboarded whose average `content_age_days` or `content_type` mix looks nothing like the training clients (the grouped-split validation in Week 6 only tells you about clients *shaped like* the ones already seen).
- `main_intent` or `content_type` missingness rates shift meaningfully from what Week 5's imputation assumed (28% missing `word_count` for keyword articles, 100% missing `main_intent` for feedly articles) — a change here means the imputation logic needs revisiting, not just a rerun.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

export_cols = ["content_id", "client_id", "archetype", "decline_risk", "reason_code", "action",
               "impressions_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days"]
ranked[export_cols].to_csv("../outputs/action_playbook_queue.csv", index=False)
print(f"Wrote {len(ranked)} rows to work/outputs/action_playbook_queue.csv (gitignored -- regenerate by rerunning)")

import json
metrics = {
    "archetype_counts": df["archetype"].value_counts().to_dict(),
    "action_counts": df["action"].value_counts().to_dict(),
    "archetype_centers": df.groupby("archetype")[cluster_features + ["decline_risk"]].mean().round(2).to_dict(orient="index"),
}
with open("../outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/playbook_metrics.json (COMMIT this one -- it's the receipt for the paper's numbers)")

fig, ax = plt.subplots(figsize=(7,4))
df["action"].value_counts().plot(kind="barh", ax=ax, color="#3b6ea5")
ax.set_xlabel("Number of pages"); ax.set_title("Action mix—Week 7 playbook")
plt.tight_layout(); plt.savefig("../figures/action_mix.svg"); plt.close()

fig, ax = plt.subplots(figsize=(7,4))
df["archetype"].value_counts().plot(kind="barh", ax=ax, color="#a53b6e")
ax.set_xlabel("Number of pages"); ax.set_title("Content archetypes (K-Means, k=4, visible pages)")
plt.tight_layout(); plt.savefig("../figures/archetype_mix.svg"); plt.close()
print("Wrote work/figures/action_mix.svg and archetype_mix.svg -- COMMIT these, the paper embeds them")

Wrote 30000 rows to work/outputs/action_playbook_queue.csv (gitignored -- regenerate by rerunning)
Wrote work/outputs/playbook_metrics.json (COMMIT this one -- it's the receipt for the paper's numbers)
Wrote work/figures/action_mix.svg and archetype_mix.svg -- COMMIT these, the paper embeds them


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.